# Whisper Colab GPU Backend

Notebook này cho phép khởi chạy Backend chuyển đổi âm thanh MP3 sang Subtitle SRT sử dụng mô hình **Whisper** của OpenAI trên GPU miễn phí của Google Colab.

### Hướng dẫn sử dụng:
1. Truy cập menu **Runtime** -> **Change runtime type** -> Chọn GPU làm Hardware accelerator.
2. Upload toàn bộ thư mục `whispercolab` lên thư mục gốc của Google Drive và mount Drive, hoặc Clone từ Github.
3. Nhấn nút **Chạy tất cả (Run all)** hoặc chạy từng ô code bên dưới theo thứ tự.
4. Chờ cho đến khi Cloudflare Tunnel khởi tạo xong, hệ thống sẽ in ra một liên kết dạng `https://xxx.trycloudflare.com`.
5. Sử dụng URL đó để gửi request `POST /transcribe` kèm theo file audio.

In [ ]:
#@title 1. Cài đặt các gói thư viện phụ thuộc (Dependencies)
import os

# Sử dụng source từ Github của bạn.
!git clone https://github.com/akavipno01/whispercolab.git /content/whispercolab

# Giả sử bạn đã upload thư mục whispercolab lên colab (hoặc thay bằng đường dẫn google drive /content/drive/MyDrive/whispercolab)
PROJECT_DIR = "/content/whispercolab"

if not os.path.exists(PROJECT_DIR):
    print(f"⚠️ Không tìm thấy thư mục {PROJECT_DIR}!")
    print("Vui lòng upload source code hoặc clone từ git.")
else:
    %cd {PROJECT_DIR}
    
    print("\n--- 1. Cài đặt ffmpeg... ---")
    !apt-get update -qq && apt-get install -y ffmpeg
    
    print("\n--- 2. Cài đặt các thư viện Python... ---")
    !pip install -r backend/requirements.txt
    
    print("\n--- 3. Tải xuống Cloudflare Tunnel... ---")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
    
    print("\nHoàn tất bước chuẩn bị môi trường!")

In [ ]:
#@title 2. Khởi chạy Backend và tạo đường truyền liên kết kết nối
import subprocess
import time
import re
import os

# Khởi động FastAPI Backend (uvicorn)
print("Đang khởi động backend Whisper...")
backend_process = subprocess.Popen(
    ["python", "run.py"],
    cwd=f"{PROJECT_DIR}/backend",
    env={
        **os.environ,
        "WHISPER_PORT": "8000",
        "WHISPER_MODEL_SIZE": "small"
    }
)

# Đợi backend uvicorn khởi động và tải mô hình Whisper vào bộ nhớ (mất khoảng 10-30 giây tuỳ mạng và model)
time.sleep(15)

# Chạy Cloudflare Tunnel
print("Đang khởi tạo Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

colab_url = None
try:
    while True:
        line = tunnel_process.stdout.readline()
        if not line:
            break
        
        # Lọc url cloudflare
        if "trycloudflare.com" in line or "error" in line.lower() or "tunnel" in line.lower():
            pass # print("[Cloudflared]", line.strip())
        
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            colab_url = match.group(0)
            print("\n" + "="*60)
            print(f"🎉 KHỞI CHẠY WHISPER BACKEND THÀNH CÔNG!")
            print(f"🔗 Địa chỉ API của bạn là:")
            print(f"   {colab_url}")
            print(f"📌 Endpoint API: POST {colab_url}/transcribe")
            print("="*60 + "\n")
            
    # Giữ cho tiến trình chạy
    backend_process.wait()
except KeyboardInterrupt:
    print("\nĐang dừng các tiến trình...")
    tunnel_process.terminate()
    backend_process.terminate()
    print("Đã dừng.")
